# Clase 7 — Computación en GPU con PyTorch

Curso **Deep Learning con Python y PyTorch desde cero** — DIH DATAlife

Duración estimada: 90 min

## Objetivos de aprendizaje

- Explicar la diferencia entre CPU (Central Processing Unit) y GPU (Graphics Processing Unit) para el entrenamiento de modelos de deep learning
- Comprobar la disponibilidad de una GPU con CUDA (Compute Unified Device Architecture)
- Mover tensores y modelos a un dispositivo con `torch.device`
- Medir la diferencia de tiempo de entrenamiento entre CPU y GPU

Este notebook funciona igual de bien con o sin GPU: cada celda comprueba el hardware disponible antes de usarlo, así que puedes ejecutarlo tal cual esté tu entorno. Para ver la diferencia real de velocidad, activa el acelerador de hardware desde **Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU** antes de llegar a la sección 5. Si lo dejas en "Ninguno" (CPU), el notebook se ejecuta igual, solo que más despacio.

## 1. CPU y GPU: qué cambia realmente

Entrenar una red neuronal consiste, sobre todo, en multiplicar matrices de números una y otra vez. Una CPU tiene pocos núcleos, muy versátiles, pensados para tareas secuenciales complejas. Una GPU tiene miles de núcleos más simples, pensados para repetir la misma operación sobre muchos datos en paralelo. Esa segunda forma de trabajar coincide exactamente con lo que necesita el entrenamiento de una red, por eso una GPU puede entrenar el mismo modelo en una fracción del tiempo que tardaría una CPU.

Para que PyTorch pueda enviar instrucciones a una GPU, usa CUDA, la plataforma de cómputo paralelo de NVIDIA. No escribes código CUDA tú mismo: PyTorch ya viene preparado para hablar ese lenguaje, y Colab tiene los controladores instalados de fábrica en cuanto activas el acelerador GPU.

In [ ]:
import torch
import time

print("Versión de PyTorch:", torch.__version__)
print("¿Hay una GPU con CUDA disponible?", torch.cuda.is_available())

## 2. Crear el `device`

`torch.cuda.is_available()` devuelve `True` o `False` según haya o no una GPU compatible con CUDA. Con ese resultado, construyes un objeto `torch.device` que usarás en todo el resto del notebook. Este patrón de una sola línea es la pieza de código más importante del capítulo: hace que el mismo código funcione en tu portátil sin GPU, en un Colab con GPU activada o en un servidor con varias GPU, sin que tengas que escribir versiones distintas.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

if torch.cuda.is_available():
    print("Nombre de la GPU:", torch.cuda.get_device_name(0))
    print("Número de GPU detectadas:", torch.cuda.device_count())
else:
    print("No hay GPU activada en este entorno de ejecución: todo el cálculo correrá en CPU")

## 3. Mover tensores a un `device`

Por defecto, cualquier tensor que crees vive en la memoria RAM (Random Access Memory) de la CPU. El método `.to(device)` lo mueve al dispositivo que le indiques. La regla que no admite excepciones: todos los tensores que participan en una misma operación deben estar en el mismo `device`, o PyTorch lanza un `RuntimeError` del tipo "expected device cpu but got device cuda:0".

In [ ]:
tensor_cpu = torch.randn(4, 4)
print("Tensor recién creado, device:", tensor_cpu.device)

tensor_en_device = tensor_cpu.to(device)
print("Tensor movido, device:", tensor_en_device.device)

# Todos los tensores de una misma operación deben estar en el mismo device
resultado = tensor_en_device + tensor_en_device
print("Operación correcta, device del resultado:", resultado.device)

# Si descomentas la siguiente línea con una GPU activada, PyTorch lanza un
# RuntimeError porque tensor_cpu sigue en la CPU y tensor_en_device ya no:
# resultado_erroneo = tensor_cpu + tensor_en_device

## 4. Mover un modelo entero al `device`

Un modelo se mueve exactamente igual que un tensor, con `.to(device)`. Aquí retomas una versión sencilla de la red de MNIST (dígitos escritos a mano) de las clases 3 y 4: dos capas ocultas con activación `ReLU` (Rectified Linear Unit) y una salida de diez clases.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class RedMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(28 * 28, 128)
        self.capa2 = nn.Linear(128, 64)
        self.capa3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)      # aplana cada imagen 28x28 en un vector de 784 valores
        x = F.relu(self.capa1(x))
        x = F.relu(self.capa2(x))
        return self.capa3(x)           # logits en bruto: CrossEntropyLoss se encarga del resto


modelo = RedMNIST().to(device)
print("Device de los parámetros del modelo:", next(modelo.parameters()).device)

## 5. Cargar MNIST real y medir el tiempo de una época

Ahora cargas el dataset MNIST real con `torchvision`, tal como en la clase 2, y entrenas una época completa. Cada lote que entrega el `DataLoader` nace en la CPU, así que lo mueves al `device` justo antes de usarlo, dentro del bucle. El modelo, en cambio, ya está en el `device` desde la celda anterior: no hace falta moverlo otra vez.

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transformaciones = transforms.Compose([transforms.ToTensor()])
dataset_entrenamiento = datasets.MNIST(
    root="./datos", train=True, download=True, transform=transformaciones
)
cargador_entrenamiento = DataLoader(dataset_entrenamiento, batch_size=128, shuffle=True)

print("Número de imágenes de entrenamiento:", len(dataset_entrenamiento))

Con los datos ya listos, entrena una única época y mide cuánto tarda con `time.time()`. Fíjate en que el bucle de entrenamiento es exactamente el mismo que en la clase 4: lo único que se ha añadido son las dos líneas `.to(device)` dentro del bucle.

In [ ]:
import torch.optim as optim

funcion_perdida = nn.CrossEntropyLoss()
optimizador = optim.Adam(modelo.parameters(), lr=0.001)

inicio = time.time()
for imagenes, etiquetas in cargador_entrenamiento:
    imagenes = imagenes.to(device)     # el lote de imágenes viaja al device
    etiquetas = etiquetas.to(device)   # las etiquetas también, o CrossEntropyLoss fallará

    optimizador.zero_grad()
    salida = modelo(imagenes)
    perdida = funcion_perdida(salida, etiquetas)
    perdida.backward()
    optimizador.step()

tiempo_epoca = time.time() - inicio
print(f"Tiempo de una época en {device}: {tiempo_epoca:.2f} segundos")

**Qué esperar de este número**: si estás en CPU, el tiempo es el que es, y sirve como referencia. Si activas GPU desde el menú de Colab y repites la sección 5 desde el principio (las celdas usan variables ya definidas, así que conviene reiniciar el entorno de ejecución primero), deberías ver una reducción notable de ese tiempo, aunque no siempre desde la primera época: la primera transferencia de datos a la GPU tiene un pequeño coste adicional que las siguientes épocas ya no pagan.

## Resumen

- Una CPU tiene pocos núcleos muy versátiles; una GPU tiene miles de núcleos simples pensados para repetir la misma operación sobre muchos datos en paralelo, justo lo que necesita el entrenamiento de una red neuronal
- CUDA es la plataforma de NVIDIA que permite a PyTorch hablar con la GPU; no escribes código CUDA tú mismo
- `torch.cuda.is_available()` te dice si hay GPU disponible; `torch.device("cuda" if torch.cuda.is_available() else "cpu")` construye un `device` que funciona con o sin GPU
- `.to(device)` mueve tensores y modelos al dispositivo elegido; todos los tensores de una misma operación deben estar en el mismo `device`
- Mueves el modelo al `device` una sola vez, fuera del bucle de entrenamiento, y cada lote de datos dentro del bucle, justo antes de usarlo
- La diferencia de tiempo entre CPU y GPU se nota sobre todo con lotes grandes y modelos con más parámetros; en el notebook de práctica de esta clase medirás también el coste de mover los datos al device